In [ ]:
# here we are going to clean beach monitoring station metadata
# and prepares beach station coordinates for mapping.

from pathlib import Path
import pandas as pd
import numpy as np

# Mount Google Drive if using Google Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

# Project folder paths
DATA_RAW = Path("/content/drive/MyDrive/STAT596/Project596_datafiles")
DATA_PROCESSED = DATA_RAW / "processed_data"
FIGURES = DATA_RAW / "figures"
TABLES = DATA_RAW / "tables"
NOTEBOOKS = DATA_RAW / "notebooks"

# Make sure organized folders exist
for folder in [DATA_PROCESSED, FIGURES, TABLES, NOTEBOOKS]:
    folder.mkdir(parents=True, exist_ok=True)

# Input files
beach_stations_file = DATA_RAW / "beach-monitoring-stations.csv"
beach_info_file = DATA_RAW / "beach-information-csv.csv"

# Output file
output_file = DATA_PROCESSED / "beach_monitoring_stations_clean.csv"

# Check input files
input_files = {
    "Beach monitoring stations": beach_stations_file,
    "Beach information": beach_info_file
}

print("Beach/station metadata setup check")
print("=" * 55)

for label, path in input_files.items():
    print(f"{label}:")
    print(f"  Path: {path}")
    print(f"  File exists: {path.exists()}")
    print()

print("Planned output:")
print(output_file)

missing_files = [label for label, path in input_files.items() if not path.exists()]

if missing_files:
    print("\nMissing required input file(s):")
    for label in missing_files:
        print(f"- {label}")
    print("\nCheck file names or file locations before continuing.")
else:
    print("\nInput check complete.")

Mounted at /content/drive
Beach/station metadata setup check
Beach monitoring stations:
  Path: /content/drive/MyDrive/STAT596/Project596_datafiles/beach-monitoring-stations.csv
  File exists: True

Beach information:
  Path: /content/drive/MyDrive/STAT596/Project596_datafiles/beach-information-csv.csv
  File exists: True

Planned output:
/content/drive/MyDrive/STAT596/Project596_datafiles/processed_data/beach_monitoring_stations_clean.csv

Input check complete.


In [ ]:
# lets review  beach and station metadata columns, shapes, missing values, and
# coordinate-related fields.

# Load raw metadata files
stations_raw = pd.read_csv(beach_stations_file)
beach_info_raw = pd.read_csv(beach_info_file)

print("Initial metadata inspection")
print("=" * 60)

# Basic shape checks
print("Beach monitoring stations file")
print("-" * 60)
print(f"Rows: {stations_raw.shape[0]}")
print(f"Columns: {stations_raw.shape[1]}")
print("\nColumn names:")
print(list(stations_raw.columns))

print("\nData types:")
print(stations_raw.dtypes)

print("\nMissing values by column:")
print(stations_raw.isna().sum().sort_values(ascending=False))

print("\nPreview:")
display(stations_raw.head())

print("\n" + "=" * 60)

print("Beach information file")
print("-" * 60)
print(f"Rows: {beach_info_raw.shape[0]}")
print(f"Columns: {beach_info_raw.shape[1]}")
print("\nColumn names:")
print(list(beach_info_raw.columns))

print("\nData types:")
print(beach_info_raw.dtypes)

print("\nMissing values by column:")
print(beach_info_raw.isna().sum().sort_values(ascending=False))

print("\nPreview:")
display(beach_info_raw.head())

# Identify likely coordinate and ID/name fields
coordinate_keywords = ["lat", "latitude", "lon", "long", "longitude", "x", "y"]
name_id_keywords = ["beach", "station", "site", "location", "id", "name"]

stations_coord_cols = [
    col for col in stations_raw.columns
    if any(keyword in col.lower() for keyword in coordinate_keywords)
]

beach_info_coord_cols = [
    col for col in beach_info_raw.columns
    if any(keyword in col.lower() for keyword in coordinate_keywords)
]

stations_name_id_cols = [
    col for col in stations_raw.columns
    if any(keyword in col.lower() for keyword in name_id_keywords)
]

beach_info_name_id_cols = [
    col for col in beach_info_raw.columns
    if any(keyword in col.lower() for keyword in name_id_keywords)
]

print("\n" + "=" * 60)
print("Likely useful columns")
print("-" * 60)

print("Station file coordinate-related columns:")
print(stations_coord_cols)

print("\nBeach info coordinate-related columns:")
print(beach_info_coord_cols)

print("\nStation file name/ID-related columns:")
print(stations_name_id_cols)

print("\nBeach info name/ID-related columns:")
print(beach_info_name_id_cols)



Initial metadata inspection
Beach monitoring stations file
------------------------------------------------------------
Rows: 1041
Columns: 61

Column names:
['Regional Board', 'Station_id', 'Station_Agency_id', 'BeachName_id', 'Station_Name', 'Station_Description', 'AgencyStationIdentifier', 'Station_UpperLat', 'Station_UpperLon', 'Station_LowerLat', 'Station_LowerLon', 'SwimSeason', 'OffSeason', 'AsNeededSeason', 'CountyCode', 'CountyName', 'Status', 'UpdateDate', 'Stations Insert Date', 'pointZero', 'Datum', 'Beach_AgencyName', 'Beach_Name', 'Description', 'BeachType', 'Beach Length', 'AttendanceWinter', 'AttendanceSummer', 'NearestCityName', 'WaterBodyName', 'WaterBodyClass', 'AB411Beach', 'USEPAID', 'BeachAccess', 'SwimSeasonLength', 'WaterBodyType', 'WaterShedName', 'Beach_UpperLat', 'Beach_UpperLon', 'Beach_LowerLat', 'Beach_LowerLon', 'Beach_County', 'Beach_User_id', 'Beach_Status', 'CountAsBeach', 'Agency_id', 'Agency_Code', 'Agency_Name', 'Agency_Jurisdiction', 'StreetAddress

,Regional Board,Station_id,Station_Agency_id,BeachName_id,Station_Name,Station_Description,AgencyStationIdentifier,Station_UpperLat,Station_UpperLon,Station_LowerLat,...,Station Contact Person,Zip,EstMonCost,EstCoastlineAdvisoryCost,EstTotalCost,Station Comments,County,Agency_User_id,Regional Board Number,Regional Board Name
0,RB8,1,21,53,0,50\' N of Santa Ana River,0,33.629290,-117.959680,33.629290,...,Michael Fennessy,92705,NaN,NaN,More than $250000,Michael Fennessy,Orange,1,8.0,Santa Ana
1,RB3,2,32,145,1000,Rincon Beach,1000,34.373398,-119.476088,34.373287,...,Richard Hauge,93009,NaN,NaN,$100000-$250000,DUNS=071818830,Ventura,1,3.0,Central Coast
2,RB4,3,32,148,10000,Solimar Beach,10000,34.310402,-119.358910,34.310402,...,Richard Hauge,93009,NaN,NaN,$100000-$250000,DUNS=071818830,Ventura,1,4.0,Los Angeles
3,RB3,4,32,145,1001,Rincon Creek (mouth),1001,34.373510,-119.475632,34.373510,...,Richard Hauge,93009,NaN,NaN,$100000-$250000,DUNS=071818830,Ventura,1,3.0,Central Coast
4,RB3,5,32,145,1050,Rincon Beach,1050,34.373900,-119.474945,34.373900,...,Richard Hauge,93009,NaN,NaN,$100000-$250000,DUNS=071818830,Ventura,1,3.0,Central Coast



Beach information file
------------------------------------------------------------
Rows: 576
Columns: 25

Column names:
['BeachName_id', 'Agency_Name', 'Beach_Name', 'Description', 'BeachType', 'Beach Length', 'AttendanceWinter', 'AttendanceSummer', 'NearestCityName', 'WaterBodyName', 'WaterBodyClass', 'AB411Beach', 'USEPAID', 'BeachAccess', 'SwimSeasonLength', 'WaterBodyType', 'WaterShedName', 'Beach_UpperLat', 'Beach_ UpperLon', 'Beach_LowerLat', 'Beach_LowerLon', 'County', 'User_id', 'Status', 'CountAsBeach']

Data types:
BeachName_id          int64
Agency_Name          object
Beach_Name           object
Description          object
BeachType            object
Beach Length        float64
AttendanceWinter      int64
AttendanceSummer      int64
NearestCityName      object
WaterBodyName        object
WaterBodyClass       object
AB411Beach           object
USEPAID              object
BeachAccess          object
SwimSeasonLength      int64
WaterBodyType        object
WaterShedName      

,BeachName_id,Agency_Name,Beach_Name,Description,BeachType,Beach Length,AttendanceWinter,AttendanceSummer,NearestCityName,WaterBodyName,...,WaterBodyType,WaterShedName,Beach_UpperLat,Beach_ UpperLon,Beach_LowerLat,Beach_LowerLon,County,User_id,Status,CountAsBeach
0,1,East Bay Regional Park District,Keller Beach (Contra Costa Co),Keller Beach,UNKNOWN,0.0500,0,0,Richmond,Pacific Ocean,...,"Sound, Bay, or Inlet",San Pablo Bay,37.921160,-122.387150,37.920530,-122.386560,East Bay Parks District,24,Active,1
1,2,Los Angeles County Department of Health Services,All_LosAngeles_County,All Beaches,UNKNOWN,42.2731,0,0,Los Angeles,Pacific Ocean,...,Open Coast,all watersheds,34.045200,-118.945000,33.707600,-118.278000,Los Angeles,74,Active,0
2,3,Los Angeles County Department of Health Services,Avalon Beach,Avalon Beach,UNKNOWN,0.4000,0,0,Avalon,Pacific Ocean,...,Open Coast,San Pedro Channel Islands,33.348152,-118.326926,33.342899,-118.323846,Los Angeles,31,Active,1
3,4,Los Angeles County Department of Health Services,Basin H,Basin H (Burton Chace Park),UNKNOWN,0.1000,0,0,Marina Del Rey,Pacific Ocean,...,"Sound, Bay, or Inlet",Santa Monica Bay,33.975940,-118.446407,33.974498,-118.446407,Los Angeles,31,Active,1
4,5,Los Angeles County Department of Health Services,Big Rock Beach,Big Rock Beach,UNKNOWN,0.8400,0,0,Malibu,Pacific Ocean,...,Open Coast,Santa Monica Bay,34.037792,-118.623733,34.036536,-118.609222,Los Angeles,31,Active,1



Likely useful columns
------------------------------------------------------------
Station file coordinate-related columns:
['Station_Agency_id', 'AgencyStationIdentifier', 'Station_UpperLat', 'Station_UpperLon', 'Station_LowerLat', 'Station_LowerLon', 'CountyCode', 'CountyName', 'Beach_AgencyName', 'BeachType', 'NearestCityName', 'WaterBodyName', 'WaterBodyClass', 'WaterBodyType', 'Beach_UpperLat', 'Beach_UpperLon', 'Beach_LowerLat', 'Beach_LowerLon', 'Beach_County', 'Agency_id', 'Agency_Code', 'Agency_Name', 'Agency_Jurisdiction', 'City', 'EstCoastlineAdvisoryCost', 'County', 'Agency_User_id']

Beach info coordinate-related columns:
['Agency_Name', 'BeachType', 'NearestCityName', 'WaterBodyName', 'WaterBodyClass', 'WaterBodyType', 'Beach_UpperLat', 'Beach_ UpperLon', 'Beach_LowerLat', 'Beach_LowerLon', 'County']

Station file name/ID-related columns:
['Station_id', 'Station_Agency_id', 'BeachName_id', 'Station_Name', 'Station_Description', 'AgencyStationIdentifier', 'Station_UpperLa

In [ ]:
# leans column names and filters beach/station metadata to San Diego County.

stations = stations_raw.copy()
beach_info = beach_info_raw.copy()

# Standardize column names by removing leading/trailing spaces
stations.columns = stations.columns.str.strip()
beach_info.columns = beach_info.columns.str.strip()

print("Column name standardization check")
print("=" * 60)

print("Station file columns with longitude/latitude terms:")
print([col for col in stations.columns if "Lat" in col or "Lon" in col])

print("\nBeach information columns with longitude/latitude terms:")
print([col for col in beach_info.columns if "Lat" in col or "Lon" in col])

# Filter to San Diego County
stations_sd = stations[
    stations["County"].astype(str).str.strip().str.lower().eq("san diego")
].copy()

beach_info_sd = beach_info[
    beach_info["County"].astype(str).str.strip().str.lower().eq("san diego")
].copy()

print("\nSan Diego County filter check")
print("=" * 60)

print(f"Station records before filter: {stations.shape[0]}")
print(f"Station records after filter:  {stations_sd.shape[0]}")

print(f"\nBeach records before filter: {beach_info.shape[0]}")
print(f"Beach records after filter:  {beach_info_sd.shape[0]}")

print("\nStation county values after filter:")
print(stations_sd["County"].value_counts(dropna=False))

print("\nBeach information county values after filter:")
print(beach_info_sd["County"].value_counts(dropna=False))

print("\nSan Diego station preview:")
display(stations_sd.head())

print("\nSan Diego beach information preview:")
display(beach_info_sd.head())



Column name standardization check
Station file columns with longitude/latitude terms:
['Station_UpperLat', 'Station_UpperLon', 'Station_LowerLat', 'Station_LowerLon', 'Beach_UpperLat', 'Beach_UpperLon', 'Beach_LowerLat', 'Beach_LowerLon']

Beach information columns with longitude/latitude terms:
['Beach_UpperLat', 'Beach_ UpperLon', 'Beach_LowerLat', 'Beach_LowerLon']

San Diego County filter check
Station records before filter: 1041
Station records after filter:  287

Beach records before filter: 576
Beach records after filter:  82

Station county values after filter:
County
San Diego    287
Name: count, dtype: int64

Beach information county values after filter:
County
San Diego    82
Name: count, dtype: int64

San Diego station preview:


,Regional Board,Station_id,Station_Agency_id,BeachName_id,Station_Name,Station_Description,AgencyStationIdentifier,Station_UpperLat,Station_UpperLon,Station_LowerLat,...,Station Contact Person,Zip,EstMonCost,EstCoastlineAdvisoryCost,EstTotalCost,Station Comments,County,Agency_User_id,Regional Board Number,Regional Board Name
87,RB9,88,23,420,All_SanDiego_County_Beaches,All_SanDiego_County_Beaches,All_SanDiego_County_Beaches,33.386900,-117.596000,33.534400,...,Keith Kezer,92112,>$250000,NaN,>$250000,Figures for sampling are yeaarly estimates for...,San Diego,1,9.0,San Diego
153,RB9,154,23,65,BAS Camp Pendleton,Camp Pendleton boat basin,BAS Camp Pendleton,33.214650,-117.199150,33.214650,...,Keith Kezer,92112,>$250000,NaN,>$250000,Figures for sampling are yeaarly estimates for...,San Diego,1,9.0,San Diego
155,RB9,156,23,76,BAY Mission Bay,Mission Bay Center,BAY Mission Bay,32.775350,-117.230050,32.214650,...,Keith Kezer,92112,>$250000,NaN,>$250000,Figures for sampling are yeaarly estimates for...,San Diego,1,9.0,San Diego
156,RB9,157,23,93,BAY SD Bay,32nd St USN,BAY SD Bay,32.680120,-117.126294,32.680120,...,Keith Kezer,92112,>$250000,NaN,>$250000,Figures for sampling are yeaarly estimates for...,San Diego,1,9.0,San Diego
157,RB9,158,23,93,BAY SD Bay 11,24th St Marine Terminal,BAY SD Bay 11,32.680087,-117.132953,32.680087,...,Keith Kezer,92112,>$250000,NaN,>$250000,Figures for sampling are yeaarly estimates for...,San Diego,1,9.0,San Diego



San Diego beach information preview:


,BeachName_id,Agency_Name,Beach_Name,Description,BeachType,Beach Length,AttendanceWinter,AttendanceSummer,NearestCityName,WaterBodyName,...,WaterBodyType,WaterShedName,Beach_UpperLat,Beach_ UpperLon,Beach_LowerLat,Beach_LowerLon,County,User_id,Status,CountAsBeach
60,64,County of San Diego Department of Environmenta...,Border Field State Park,Border Field State Park,UNKNOWN,1.29,0,0,San Diego,Pacific Ocean,...,Open Coast,Cottonwood Creek-Tijuana River,32.552820,-117.127708,32.534367,-117.124197,San Diego,4,Active,1
61,65,County of San Diego Department of Environmenta...,USMC Camp Pendleton,Camp Del Mar (USMC Camp Pendleton),UNKNOWN,9.91,0,0,USMC Camp Pendleton,Pacific Ocean,...,Open Coast,Aliso Creek-San Onofre Creek,33.331617,-117.503883,33.214245,-117.405285,San Diego,4,Active,1
62,66,County of San Diego Department of Environmenta...,Cardiff State Beach,Cardiff State Beach,UNKNOWN,1.13,0,0,Del Mar,Pacific Ocean,...,Open Coast,Escondido Creek-San Luis Rey River,33.015560,-117.281346,32.999470,-117.277931,San Diego,4,Active,1
63,67,County of San Diego Department of Environmenta...,Carlsbad municipal beach,Carlsbad City Beach,UNKNOWN,0.75,0,0,Carlsbad,Pacific Ocean,...,Open Coast,Escondido Creek-San Luis Rey River,33.165100,-117.359400,33.155900,-117.352600,San Diego,147,Active,1
64,68,County of San Diego Department of Environmenta...,Carlsbad State Beach,Carlsbad State Beach,UNKNOWN,1.76,0,0,Carlsbad,Pacific Ocean,...,Open Coast,Escondido Creek-San Luis Rey River,33.155900,-117.352600,33.133800,-117.337500,San Diego,147,Active,1



County filtering complete.


In [ ]:
# Verifies San Diego beach/station latitude and longitude fields

beach_info_sd = beach_info_sd.rename(columns={"Beach_ UpperLon": "Beach_UpperLon"})

# Coordinate columns to check
station_coord_cols = [
    "Station_UpperLat", "Station_UpperLon",
    "Station_LowerLat", "Station_LowerLon",
    "Beach_UpperLat", "Beach_UpperLon",
    "Beach_LowerLat", "Beach_LowerLon"
]

beach_coord_cols = [
    "Beach_UpperLat", "Beach_UpperLon",
    "Beach_LowerLat", "Beach_LowerLon"
]

# Convert coordinate columns to numeric
for col in station_coord_cols:
    if col in stations_sd.columns:
        stations_sd[col] = pd.to_numeric(stations_sd[col], errors="coerce")

for col in beach_coord_cols:
    if col in beach_info_sd.columns:
        beach_info_sd[col] = pd.to_numeric(beach_info_sd[col], errors="coerce")

print("Coordinate column check")
print("=" * 60)

print("Station coordinate columns:")
print([col for col in station_coord_cols if col in stations_sd.columns])

print("\nBeach information coordinate columns:")
print([col for col in beach_coord_cols if col in beach_info_sd.columns])

print("\nMissing station coordinates:")
print(stations_sd[station_coord_cols].isna().sum())

print("\nMissing beach information coordinates:")
print(beach_info_sd[beach_coord_cols].isna().sum())

# Check coordinate ranges for San Diego County
print("\nStation coordinate ranges:")
for col in station_coord_cols:
    if col in stations_sd.columns:
        print(f"{col}: min={stations_sd[col].min()}, max={stations_sd[col].max()}")

print("\nBeach information coordinate ranges:")
for col in beach_coord_cols:
    if col in beach_info_sd.columns:
        print(f"{col}: min={beach_info_sd[col].min()}, max={beach_info_sd[col].max()}")

# Check potential duplicate identifiers
print("\nDuplicate checks")
print("=" * 60)

print(f"Duplicate Station_id values: {stations_sd['Station_id'].duplicated().sum()}")
print(f"Duplicate Station_Agency_id values: {stations_sd['Station_Agency_id'].duplicated().sum()}")
print(f"Duplicate BeachName_id values in station file: {stations_sd['BeachName_id'].duplicated().sum()}")
print(f"Duplicate BeachName_id values in beach information file: {beach_info_sd['BeachName_id'].duplicated().sum()}")

print("\nUnique count checks")
print("=" * 60)
print(f"Unique stations: {stations_sd['Station_id'].nunique()}")
print(f"Unique beaches in station file: {stations_sd['BeachName_id'].nunique()}")
print(f"Unique beaches in beach information file: {beach_info_sd['BeachName_id'].nunique()}")

print("\nSample station coordinate fields:")
display(stations_sd[
    ["Station_id", "Station_Name", "BeachName_id", "Beach_Name",
     "Station_UpperLat", "Station_UpperLon", "Station_LowerLat", "Station_LowerLon",
     "Beach_UpperLat", "Beach_UpperLon", "Beach_LowerLat", "Beach_LowerLon"]
].head(10))



Coordinate column check
Station coordinate columns:
['Station_UpperLat', 'Station_UpperLon', 'Station_LowerLat', 'Station_LowerLon', 'Beach_UpperLat', 'Beach_UpperLon', 'Beach_LowerLat', 'Beach_LowerLon']

Beach information coordinate columns:
['Beach_UpperLat', 'Beach_UpperLon', 'Beach_LowerLat', 'Beach_LowerLon']

Missing station coordinates:
Station_UpperLat    12
Station_UpperLon    12
Station_LowerLat    85
Station_LowerLon    85
Beach_UpperLat       8
Beach_UpperLon       8
Beach_LowerLat       8
Beach_LowerLon       8
dtype: int64

Missing beach information coordinates:
Beach_UpperLat    6
Beach_UpperLon    6
Beach_LowerLat    6
Beach_LowerLon    6
dtype: int64

Station coordinate ranges:
Station_UpperLat: min=32.4191, max=33.3869
Station_UpperLon: min=-117.596, max=117.596
Station_LowerLat: min=32.21465, max=33.5344
Station_LowerLon: min=-117.5952, max=117.278472
Beach_UpperLat: min=32.55282, max=33.3869
Beach_UpperLon: min=-117.59623, max=-117.06321
Beach_LowerLat: min=32.1327

,Station_id,Station_Name,BeachName_id,Beach_Name,Station_UpperLat,Station_UpperLon,Station_LowerLat,Station_LowerLon,Beach_UpperLat,Beach_UpperLon,Beach_LowerLat,Beach_LowerLon
87,88,All_SanDiego_County_Beaches,420,All_SanDiego_County,33.386900,-117.596000,33.534400,-117.124000,33.386900,-117.596000,32.534400,-117.124000
153,154,BAS Camp Pendleton,65,USMC Camp Pendleton,33.214650,-117.199150,33.214650,-117.199150,33.331617,-117.503883,33.214245,-117.405285
155,156,BAY Mission Bay,76,Mission Bay,32.775350,-117.230050,32.214650,-117.230050,32.760031,-117.247919,32.761722,-117.241383
156,157,BAY SD Bay,93,San Diego Bay,32.680120,-117.126294,32.680120,-117.126294,32.707794,-117.237025,32.703083,-117.180908
157,158,BAY SD Bay 11,93,San Diego Bay,32.680087,-117.132953,32.680087,-117.132953,32.707794,-117.237025,32.703083,-117.180908
158,159,BAY SD Bay11,93,San Diego Bay,32.722981,-117.188651,32.722981,-117.188651,32.707794,-117.237025,32.703083,-117.180908
159,160,BAY SD Bay12,93,San Diego Bay,32.730431,-117.213900,32.730431,-117.213900,32.707794,-117.237025,32.703083,-117.180908
160,161,BAY SD Bay2,93,San Diego Bay,32.712515,-117.211663,32.712511,-117.211663,32.707794,-117.237025,32.703083,-117.180908
161,162,BAY SD Bay3,93,San Diego Bay,32.717576,-117.231982,32.717576,-117.231982,32.707794,-117.237025,32.703083,-117.180908
162,163,BAY SD Bay4,93,San Diego Bay,32.673485,-117.166872,32.673485,-117.166872,32.707794,-117.237025,32.703083,-117.180908



Coordinate QA complete.


In [ ]:
stations_clean = stations_sd.copy()
beach_info_clean = beach_info_sd.copy()

# Longitude columns to clean
station_lon_cols = ["Station_UpperLon", "Station_LowerLon", "Beach_UpperLon", "Beach_LowerLon"]
beach_lon_cols = ["Beach_UpperLon", "Beach_LowerLon"]

# Save original longitude values for QA
for col in station_lon_cols:
    stations_clean[f"{col}_original"] = stations_clean[col]

for col in beach_lon_cols:
    beach_info_clean[f"{col}_original"] = beach_info_clean[col]

# Correct longitude sign errors for San Diego-area records
# San Diego longitudes should be negative, roughly around -117.
for col in station_lon_cols:
    stations_clean[col] = np.where(
        stations_clean[col] > 0,
        -stations_clean[col],
        stations_clean[col]
    )

for col in beach_lon_cols:
    beach_info_clean[col] = np.where(
        beach_info_clean[col] > 0,
        -beach_info_clean[col],
        beach_info_clean[col]
    )

# Count corrected longitude values
print("Longitude sign correction check")
print("=" * 60)

for col in station_lon_cols:
    corrected_count = (
        stations_clean[f"{col}_original"].notna()
        & stations_clean[col].notna()
        & (stations_clean[f"{col}_original"] != stations_clean[col])
    ).sum()
    print(f"{col} corrected values: {corrected_count}")

print()

for col in beach_lon_cols:
    corrected_count = (
        beach_info_clean[f"{col}_original"].notna()
        & beach_info_clean[col].notna()
        & (beach_info_clean[f"{col}_original"] != beach_info_clean[col])
    ).sum()
    print(f"{col} corrected values: {corrected_count}")

# Create working station point coordinates
# Prefer Station_Upper coordinates, then Station_Lower coordinates, then beach midpoint coordinates.
stations_clean["beach_mid_lat"] = stations_clean[["Beach_UpperLat", "Beach_LowerLat"]].mean(axis=1)
stations_clean["beach_mid_lon"] = stations_clean[["Beach_UpperLon", "Beach_LowerLon"]].mean(axis=1)

stations_clean["station_lat"] = stations_clean["Station_UpperLat"]
stations_clean["station_lon"] = stations_clean["Station_UpperLon"]

stations_clean["station_lat"] = stations_clean["station_lat"].fillna(stations_clean["Station_LowerLat"])
stations_clean["station_lon"] = stations_clean["station_lon"].fillna(stations_clean["Station_LowerLon"])

stations_clean["station_lat"] = stations_clean["station_lat"].fillna(stations_clean["beach_mid_lat"])
stations_clean["station_lon"] = stations_clean["station_lon"].fillna(stations_clean["beach_mid_lon"])

# Create coordinate source label
stations_clean["coordinate_source"] = np.select(
    [
        stations_clean["Station_UpperLat"].notna() & stations_clean["Station_UpperLon"].notna(),
        stations_clean["Station_LowerLat"].notna() & stations_clean["Station_LowerLon"].notna(),
        stations_clean["beach_mid_lat"].notna() & stations_clean["beach_mid_lon"].notna()
    ],
    [
        "station_upper",
        "station_lower",
        "beach_midpoint"
    ],
    default="missing"
)

# Coordinate range checks for San Diego County
stations_clean["valid_station_point"] = (
    stations_clean["station_lat"].between(32.0, 34.0)
    & stations_clean["station_lon"].between(-118.0, -116.0)
)

beach_info_clean["beach_mid_lat"] = beach_info_clean[["Beach_UpperLat", "Beach_LowerLat"]].mean(axis=1)
beach_info_clean["beach_mid_lon"] = beach_info_clean[["Beach_UpperLon", "Beach_LowerLon"]].mean(axis=1)

beach_info_clean["valid_beach_point"] = (
    beach_info_clean["beach_mid_lat"].between(32.0, 34.0)
    & beach_info_clean["beach_mid_lon"].between(-118.0, -116.0)
)

print("\nWorking coordinate QA")
print("=" * 60)

print("Station coordinate source counts:")
print(stations_clean["coordinate_source"].value_counts(dropna=False))

print("\nValid station point counts:")
print(stations_clean["valid_station_point"].value_counts(dropna=False))

print("\nValid beach midpoint counts:")
print(beach_info_clean["valid_beach_point"].value_counts(dropna=False))

print("\nStation working coordinate ranges:")
print(f"station_lat: min={stations_clean['station_lat'].min()}, max={stations_clean['station_lat'].max()}")
print(f"station_lon: min={stations_clean['station_lon'].min()}, max={stations_clean['station_lon'].max()}")

print("\nBeach midpoint coordinate ranges:")
print(f"beach_mid_lat: min={beach_info_clean['beach_mid_lat'].min()}, max={beach_info_clean['beach_mid_lat'].max()}")
print(f"beach_mid_lon: min={beach_info_clean['beach_mid_lon'].min()}, max={beach_info_clean['beach_mid_lon'].max()}")

print("\nRecords with invalid or missing working station coordinates:")
display(
    stations_clean.loc[
        ~stations_clean["valid_station_point"],
        ["Station_id", "Station_Name", "BeachName_id", "Beach_Name",
         "station_lat", "station_lon", "coordinate_source"]
    ]
)



Longitude sign correction check
Station_UpperLon corrected values: 1
Station_LowerLon corrected values: 1
Beach_UpperLon corrected values: 0
Beach_LowerLon corrected values: 0

Beach_UpperLon corrected values: 0
Beach_LowerLon corrected values: 0

Working coordinate QA
Station coordinate source counts:
coordinate_source
station_upper     275
beach_midpoint      9
missing             3
Name: count, dtype: int64

Valid station point counts:
valid_station_point
True     284
False      3
Name: count, dtype: int64

Valid beach midpoint counts:
valid_beach_point
True     76
False     6
Name: count, dtype: int64

Station working coordinate ranges:
station_lat: min=32.4191, max=33.3869
station_lon: min=-117.596, max=-117.044857

Beach midpoint coordinate ranges:
beach_mid_lat: min=32.35655, max=33.3592335
beach_mid_lon: min=-117.55005650000001, max=-117.05403

Records with invalid or missing working station coordinates:


,Station_id,Station_Name,BeachName_id,Beach_Name,station_lat,station_lon,coordinate_source
249,253,Central_SanDiego_County_Beaches,421,Central County Beaches,NaN,NaN,missing
536,550,NandC_SanDiego_County_Beaches,428,north and cntrl county beaches,NaN,NaN,missing
701,715,Southern_SanDiego_County_Beaches,431,south county beaches,NaN,NaN,missing



Coordinate cleaning complete.


In [ ]:
# it shows that we have 3 records missing so I want to make sure they arent too
# important

invalid_station_points = stations_clean.loc[
    ~stations_clean["valid_station_point"]
].copy()

print("Invalid or missing station point records")
print("=" * 70)

print(f"Invalid/missing station records: {invalid_station_points.shape[0]}")
print()

invalid_cols = [
    "Station_id", "Station_Name", "Station_Description",
    "BeachName_id", "Beach_Name", "County",
    "Station_UpperLat", "Station_UpperLon",
    "Station_LowerLat", "Station_LowerLon",
    "Beach_UpperLat", "Beach_UpperLon",
    "Beach_LowerLat", "Beach_LowerLon",
    "beach_mid_lat", "beach_mid_lon",
    "station_lat", "station_lon",
    "coordinate_source", "valid_station_point"
]

display(invalid_station_points[invalid_cols])

print("\nCoordinate source counts for invalid records:")
print(invalid_station_points["coordinate_source"].value_counts(dropna=False))

print("\nMissing values in invalid records:")
print(invalid_station_points[[
    "Station_UpperLat", "Station_UpperLon",
    "Station_LowerLat", "Station_LowerLon",
    "Beach_UpperLat", "Beach_UpperLon",
    "Beach_LowerLat", "Beach_LowerLon",
    "station_lat", "station_lon"
]].isna().sum())



Invalid or missing station point records
Invalid/missing station records: 3



,Station_id,Station_Name,Station_Description,BeachName_id,Beach_Name,County,Station_UpperLat,Station_UpperLon,Station_LowerLat,Station_LowerLon,Beach_UpperLat,Beach_UpperLon,Beach_LowerLat,Beach_LowerLon,beach_mid_lat,beach_mid_lon,station_lat,station_lon,coordinate_source,valid_station_point
249,253,Central_SanDiego_County_Beaches,Torrey Pines to Pt Loma,421,Central County Beaches,San Diego,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,missing,False
536,550,NandC_SanDiego_County_Beaches,San Onofre to Pt Loma,428,north and cntrl county beaches,San Diego,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,missing,False
701,715,Southern_SanDiego_County_Beaches,US/ Mex border to Pt Loma,431,south county beaches,San Diego,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,missing,False



Coordinate source counts for invalid records:
coordinate_source
missing    3
Name: count, dtype: int64

Missing values in invalid records:
Station_UpperLat    3
Station_UpperLon    3
Station_LowerLat    3
Station_LowerLon    3
Beach_UpperLat      3
Beach_UpperLon      3
Beach_LowerLat      3
Beach_LowerLon      3
station_lat         3
station_lon         3
dtype: int64

Invalid record check complete.


In [ ]:
# the missing point also have no coordinates so we cant use them
# removing it to make data clean
# lets builds the final San Diego beach monitoring station metadata table
# with usable coordinates.


stations_final = stations_clean.loc[
    stations_clean["valid_station_point"]
].copy()

# Select columns useful for mapping, joining, and documentation
final_columns = [
    "Station_id",
    "Station_Agency_id",
    "BeachName_id",
    "Station_Name",
    "Station_Description",
    "AgencyStationIdentifier",
    "Beach_Name",
    "BeachType",
    "NearestCityName",
    "WaterBodyName",
    "WaterBodyClass",
    "WaterBodyType",
    "WaterShedName",
    "AB411Beach",
    "USEPAID",
    "BeachAccess",
    "County",
    "CountyName",
    "Status",
    "Beach_Status",
    "Agency_Name",
    "Agency_Jurisdiction",
    "Regional Board",
    "Regional Board Name",
    "station_lat",
    "station_lon",
    "coordinate_source",
    "Beach_UpperLat",
    "Beach_UpperLon",
    "Beach_LowerLat",
    "Beach_LowerLon",
    "beach_mid_lat",
    "beach_mid_lon",
    "valid_station_point"
]

# Keep only columns that exist in the dataframe
final_columns = [col for col in final_columns if col in stations_final.columns]

stations_final = stations_final[final_columns].copy()

# Rename columns to consistent snake_case
stations_final = stations_final.rename(columns={
    "Station_id": "station_id",
    "Station_Agency_id": "station_agency_id",
    "BeachName_id": "beach_name_id",
    "Station_Name": "station_name",
    "Station_Description": "station_description",
    "AgencyStationIdentifier": "agency_station_identifier",
    "Beach_Name": "beach_name",
    "BeachType": "beach_type",
    "NearestCityName": "nearest_city",
    "WaterBodyName": "water_body_name",
    "WaterBodyClass": "water_body_class",
    "WaterBodyType": "water_body_type",
    "WaterShedName": "watershed_name",
    "AB411Beach": "ab411_beach",
    "USEPAID": "usepa_id",
    "BeachAccess": "beach_access",
    "County": "county",
    "CountyName": "county_name",
    "Status": "station_status",
    "Beach_Status": "beach_status",
    "Agency_Name": "agency_name",
    "Agency_Jurisdiction": "agency_jurisdiction",
    "Regional Board": "regional_board",
    "Regional Board Name": "regional_board_name",
    "Beach_UpperLat": "beach_upper_lat",
    "Beach_UpperLon": "beach_upper_lon",
    "Beach_LowerLat": "beach_lower_lat",
    "Beach_LowerLon": "beach_lower_lon"
})

# Standardize text columns
text_cols = stations_final.select_dtypes(include="object").columns

for col in text_cols:
    stations_final[col] = (
        stations_final[col]
        .astype(str)
        .str.strip()
        .replace({"nan": np.nan, "None": np.nan})
    )

# Sort for readability
stations_final = stations_final.sort_values(
    by=["beach_name", "station_name", "station_id"],
    na_position="last"
).reset_index(drop=True)

print("Final clean station metadata QA")
print("=" * 70)

print(f"Original San Diego station records: {stations_sd.shape[0]}")
print(f"Removed unmappable records: {(~stations_clean['valid_station_point']).sum()}")
print(f"Final clean station records: {stations_final.shape[0]}")
print(f"Final columns: {stations_final.shape[1]}")

print("\nMissing values in key fields:")
key_fields = [
    "station_id", "beach_name_id", "station_name", "beach_name",
    "station_lat", "station_lon", "coordinate_source"
]
print(stations_final[key_fields].isna().sum())

print("\nCoordinate source counts:")
print(stations_final["coordinate_source"].value_counts(dropna=False))

print("\nCoordinate ranges:")
print(f"station_lat: min={stations_final['station_lat'].min()}, max={stations_final['station_lat'].max()}")
print(f"station_lon: min={stations_final['station_lon'].min()}, max={stations_final['station_lon'].max()}")

print("\nDuplicate station_id values:")
print(stations_final["station_id"].duplicated().sum())

print("\nUnique beaches represented in final station metadata:")
print(stations_final["beach_name_id"].nunique())

print("\nPreview of final clean station metadata:")
display(stations_final.head(10))



Final clean station metadata QA
Original San Diego station records: 287
Removed unmappable records: 3
Final clean station records: 284
Final columns: 34

Missing values in key fields:
station_id           0
beach_name_id        0
station_name         0
beach_name           0
station_lat          0
station_lon          0
coordinate_source    0
dtype: int64

Coordinate source counts:
coordinate_source
station_upper     275
beach_midpoint      9
Name: count, dtype: int64

Coordinate ranges:
station_lat: min=32.4191, max=33.3869
station_lon: min=-117.596, max=-117.044857

Duplicate station_id values:
0

Unique beaches represented in final station metadata:
74

Preview of final clean station metadata:


,station_id,station_agency_id,beach_name_id,station_name,station_description,agency_station_identifier,beach_name,beach_type,nearest_city,water_body_name,...,station_lat,station_lon,coordinate_source,beach_upper_lat,beach_upper_lon,beach_lower_lat,beach_lower_lon,beach_mid_lat,beach_mid_lon,valid_station_point
0,277,23,413,CRK Agua Hedionda,Agua Hedionda Lagoon,CRK Agua Hedionda,Agua Hedionda Lagoon,UNKNOWN,Del Mar,Pacific Ocean,...,33.144612,-117.326613,beach_midpoint,33.147393,-117.333249,33.141831,-117.319977,33.144612,-117.326613,True
1,385,23,413,EH-450,Snug Harbor swimming area,EH-450,Agua Hedionda Lagoon,UNKNOWN,Del Mar,Pacific Ocean,...,33.145000,-117.340900,station_upper,33.147393,-117.333249,33.141831,-117.319977,33.144612,-117.326613,True
2,386,23,413,EH-452,Magdalena Ecke YMCA Park,EH-452,Agua Hedionda Lagoon,UNKNOWN,Del Mar,Pacific Ocean,...,33.144618,-117.337290,station_upper,33.147393,-117.333249,33.141831,-117.319977,33.144612,-117.326613,True
3,472,23,413,LAG Agua Hedionda,Agua Hedionda Lagoon,LAG Agua Hedionda,Agua Hedionda Lagoon,UNKNOWN,Del Mar,Pacific Ocean,...,33.139292,-117.329032,station_upper,33.147393,-117.333249,33.141831,-117.319977,33.144612,-117.326613,True
4,88,23,420,All_SanDiego_County_Beaches,All_SanDiego_County_Beaches,All_SanDiego_County_Beaches,All_SanDiego_County,UNKNOWN,San Diego,Pacific Ocean,...,33.386900,-117.596000,station_upper,33.386900,-117.596000,32.534400,-117.124000,32.960650,-117.360000,True
5,179,23,382,BC-010,Playas Blanca MEX,BC-010,"Baja California, MEXICO",UNKNOWN,Tijuana,Pacific Ocean,...,32.419100,-117.097300,station_upper,NaN,NaN,NaN,NaN,NaN,NaN,True
6,180,23,382,BC-020,El Vigia MEX,BC-020,"Baja California, MEXICO",UNKNOWN,Tijuana,Pacific Ocean,...,32.498700,-117.123000,station_upper,NaN,NaN,NaN,NaN,NaN,NaN,True
7,181,23,382,BC-030,Playas De Tijuana MEX,BC-030,"Baja California, MEXICO",UNKNOWN,Tijuana,Pacific Ocean,...,32.525700,-117.124000,station_upper,NaN,NaN,NaN,NaN,NaN,NaN,True
8,455,23,64,IB-010,Border Fence N side,IB-010,Border Field State Park,UNKNOWN,San Diego,Pacific Ocean,...,32.535950,-117.124300,station_upper,32.552820,-117.127708,32.534367,-117.124197,32.543594,-117.125952,True
9,456,23,64,IB-020,Monument Rd.,IB-020,Border Field State Park,UNKNOWN,San Diego,Pacific Ocean,...,32.543410,-117.125228,station_upper,32.552820,-117.127708,32.534367,-117.124197,32.543594,-117.125952,True



Final clean station metadata table prepared.


In [ ]:
# as usual lets do a quality assurance summary (qa)
stations_final.to_csv(output_file, index=False)

# Create QA summary output
metadata_summary = pd.DataFrame([
    {
        "dataset": "beach_monitoring_stations_clean",
        "original_san_diego_station_records": stations_sd.shape[0],
        "removed_unmappable_records": int((~stations_clean["valid_station_point"]).sum()),
        "final_clean_station_records": stations_final.shape[0],
        "final_columns": stations_final.shape[1],
        "unique_station_ids": stations_final["station_id"].nunique(),
        "duplicate_station_ids": stations_final["station_id"].duplicated().sum(),
        "unique_beaches_represented": stations_final["beach_name_id"].nunique(),
        "missing_station_lat": stations_final["station_lat"].isna().sum(),
        "missing_station_lon": stations_final["station_lon"].isna().sum(),
        "min_station_lat": stations_final["station_lat"].min(),
        "max_station_lat": stations_final["station_lat"].max(),
        "min_station_lon": stations_final["station_lon"].min(),
        "max_station_lon": stations_final["station_lon"].max(),
        "station_upper_coordinate_records": int((stations_final["coordinate_source"] == "station_upper").sum()),
        "beach_midpoint_coordinate_records": int((stations_final["coordinate_source"] == "beach_midpoint").sum())
    }
])

metadata_summary_file = TABLES / "beach_monitoring_stations_metadata_summary.csv"
metadata_summary.to_csv(metadata_summary_file, index=False)

# Reload saved file for verification
stations_reloaded = pd.read_csv(output_file)

print("Saved beach monitoring station metadata")
print("=" * 70)

print(f"Processed file path: {output_file}")
print(f"File exists: {output_file.exists()}")
print()

print(f"Summary table path: {metadata_summary_file}")
print(f"File exists: {metadata_summary_file.exists()}")

print("\nReloaded file QA")
print("=" * 70)
print(f"Rows: {stations_reloaded.shape[0]}")
print(f"Columns: {stations_reloaded.shape[1]}")
print(f"Duplicate station_id values: {stations_reloaded['station_id'].duplicated().sum()}")

print("\nMissing values in key fields:")
print(stations_reloaded[
    ["station_id", "beach_name_id", "station_name", "beach_name", "station_lat", "station_lon"]
].isna().sum())

print("\nCoordinate source counts:")
print(stations_reloaded["coordinate_source"].value_counts(dropna=False))

print("\nQA summary:")
display(metadata_summary)



Saved beach monitoring station metadata
Processed file path: /content/drive/MyDrive/STAT596/Project596_datafiles/processed_data/beach_monitoring_stations_clean.csv
File exists: True

Summary table path: /content/drive/MyDrive/STAT596/Project596_datafiles/tables/beach_monitoring_stations_metadata_summary.csv
File exists: True

Reloaded file QA
Rows: 284
Columns: 34
Duplicate station_id values: 0

Missing values in key fields:
station_id       0
beach_name_id    0
station_name     0
beach_name       0
station_lat      0
station_lon      0
dtype: int64

Coordinate source counts:
coordinate_source
station_upper     275
beach_midpoint      9
Name: count, dtype: int64

QA summary:


,dataset,original_san_diego_station_records,removed_unmappable_records,final_clean_station_records,final_columns,unique_station_ids,duplicate_station_ids,unique_beaches_represented,missing_station_lat,missing_station_lon,min_station_lat,max_station_lat,min_station_lon,max_station_lon,station_upper_coordinate_records,beach_midpoint_coordinate_records
0,beach_monitoring_stations_clean,287,3,284,34,284,0,74,0,0,32.4191,33.3869,-117.596,-117.044857,275,9



Save check complete.
